##Seção 1 (Markdown) — Cabeçalho do Desafio

##Desafio: Resgatar Palavras Perdidas em Livros Antigos 🎬

Imagine uma descoberta histórica: pesquisadores da língua portuguesa encontram um manuscrito raríssimo, uma janela para o passado. O problema? O tempo foi cruel, e muitas de suas páginas estão corroídas, deixando lacunas na história.
É aqui que sua jornada começa. Como especialista em IA, você tem a chance de dar vida nova a essa obra, treinando um modelo para prever os trechos faltantes.
Felizmente, as ferramentas para essa expedição já estão prontas. Seu papel não é o de construtor, mas o de guia: você irá escolher os melhores caminhos para levar o modelo até o tesouro do conhecimento perdido. ⛏️

##O que você vai fazer
- Analisar os dados de textos com lacunas ocultas.
- Escolher e instanciar um Tokenizador para processar as palavras.
- Escolher um modelo de Redes Neurais ou Processamento de Linguagem Natural.
- Gerar o arquivo final de submissão correspondente ao conjunto de teste.

##Regras
- ⚠️ Células marcadas com "não altere esta célula" devem ser mantidas intactas.
- 💻 Células marcadas com "SEU CÓDIGO AQUI" são onde você deve implementar sua solução.
- 🛑 Não altere o nome do arquivo de saída (`submission.csv`) e não utilize `google.colab` para download.


##Seção 2 (Código) — Célula de Configuração

In [ ]:
# ============================================================
# CONFIGURAÇÃO — não altere esta célula
# ============================================================

# 1. Instalação de dependências
import subprocess
import sys

def instalar(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import gdown
except ImportError:
    instalar("gdown")
    import gdown

# 2. Imports padrão
import os
import random
import warnings
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from transformers import BertTokenizer, GPT2Tokenizer, T5Tokenizer

warnings.filterwarnings("ignore")

# 3. Seed para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 4. Paths padrão (nunca hardcode caminhos fora desta célula)
DADOS_DIR = "data"
OUTPUT_FILE = "submission.csv"

# 5. Device (CPU/GPU automático)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f" Configuração OK | Device: {DEVICE} | Seed: {SEED}")

##Seção 3 (Código) — Célula de Carregamento de Dados

In [ ]:
# ============================================================
# CARREGAMENTO DE DADOS — não altere esta célula
# ============================================================

def get_direct_download_link(drive_url):
    file_id = drive_url.split("/d/")[1].split("/")[0]
    return f"https://drive.google.com/uc?id={file_id}"

def load_data_from_drive(train_url, test_url, sample_submission_url, pasta_destino: str = DADOS_DIR):
    """
    Baixa do Google Drive e carrega os dados da competição.
    """
    print("Baixando dados do Google Drive...")
    os.makedirs(pasta_destino, exist_ok=True)

    train_direct_url = get_direct_download_link(train_url)
    test_direct_url = get_direct_download_link(test_url)
    sample_direct_url = get_direct_download_link(sample_submission_url)

    print("Baixando arquivo de treino...")
    gdown.download(train_direct_url, os.path.join(pasta_destino, "train_raw.csv"), quiet=False)

    print("Baixando arquivo de teste...")
    gdown.download(test_direct_url, os.path.join(pasta_destino, "test_raw.csv"), quiet=False)

    print("Baixando sample_submission...")
    gdown.download(sample_direct_url, os.path.join(pasta_destino, "sample_submission.csv"), quiet=False)

    train_df = pd.read_csv(os.path.join(pasta_destino, "train_raw.csv"))
    test_df = pd.read_csv(os.path.join(pasta_destino, "test_raw.csv"))
    sample_submission_df = pd.read_csv(os.path.join(pasta_destino, "sample_submission.csv"))

    return train_df, test_df, sample_submission_df

# Executar carregamento de dados oficiais
TRAIN_URL = 'https://drive.google.com/file/d/1B80-_TovecH6YDmB_dB19qfxp7ACuoNk/view?usp=drive_link'
TEST_URL = 'https://drive.google.com/file/d/1JVERiYZvOHcwCC_wyFYRbOrpMRCN1rT3/view?usp=drive_link'
SAMPLE_SUBMISSION_URL = 'https://drive.google.com/file/d/1lyjzRVHsdQrCXXVfp_skI7TzohYGbEAn/view?usp=drive_link'

train_df, test_df, sample_submission = load_data_from_drive(TRAIN_URL, TEST_URL, SAMPLE_SUBMISSION_URL)

# Divisão inicial em subsets
train_subset, val_subset = train_test_split(train_df, test_size=0.2, random_state=SEED, shuffle=True)
train_texts = train_subset["masked_text"].astype(str).tolist()
val_texts   = val_subset["masked_text"].astype(str).tolist()

vectorizer = CountVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_texts)
X_val   = vectorizer.transform(val_texts)
bow_input_size = X_train.shape[1]

# Visualização inicial padronizada (obrigatório manter)
print("\n--- Primeiras linhas ---")
display(train_df.head())
print('\n--- Tipos e nulos ---')
display(train_df.info())

##Seção 4 (Código) — Funções Auxiliares (Processamento e Modelagem)

In [ ]:
# ============================================================
# FUNÇÕES AUXILIARES — não altere esta célula
# ============================================================

def get_wordpiece_tokenizer():
    print("Carregando WordPiece Tokenizer (BERT)...")
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    tokenizer.mask_token = "<MASK>"
    return tokenizer

def get_bpe_tokenizer():
    print("Carregando BPE Tokenizer (GPT-2)...")
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.mask_token = "<MASK>"
    return tokenizer

def get_sentencepiece_tokenizer():
    print("Carregando SentencePiece Tokenizer (T5)...")
    tokenizer = T5Tokenizer.from_pretrained("t5-small")
    tokenizer.mask_token = "<MASK>"
    return tokenizer

def tokenizer_turbo():
    class SabotagedTokenizer:
        def __init__(self):
            self.mask_token = "<MASK>"
            self.unk_token_id = 100
        def tokenize(self, text):
            return ["[SABOTAGE]"] * len(text.split())
        def __call__(self, text, truncation=True, padding='max_length', max_length=128, return_tensors='pt', **kwargs):
            num_words = len(text.split())
            fake_tokens = [101]
            for _ in range(min(num_words, max_length-2)):
                fake_tokens.append(999)
            fake_tokens.append(102)
            while len(fake_tokens) < max_length:
                fake_tokens.append(0)
            if len(fake_tokens) > max_length:
                fake_tokens = fake_tokens[:max_length-1] + [102]
            if return_tensors == 'pt':
                return {
                    'input_ids': torch.tensor([fake_tokens]),
                    'attention_mask': torch.tensor([[1 if token != 0 else 0 for token in fake_tokens]])
                }
            else:
                return {
                    'input_ids': [fake_tokens],
                    'attention_mask': [[1 if token != 0 else 0 for token in fake_tokens]]
                }
        def encode(self, text, add_special_tokens=False):
            tokens = [999] * len(text.split())
            if add_special_tokens:
                tokens = [101] + tokens + [102]
            return tokens
    return SabotagedTokenizer()

def get_tokenizer(tokenizer_name):
    if tokenizer_name == "WordPiece":
        return get_wordpiece_tokenizer()
    elif tokenizer_name == "Byte Pair Encoding":
        return get_bpe_tokenizer()
    elif tokenizer_name == "SentencePiece":
        return get_sentencepiece_tokenizer()
    elif tokenizer_name == "Turbo":
        return tokenizer_turbo()
    else:
        return get_sentencepiece_tokenizer()

class IMDBTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128, model_type="neural", is_test=False):
        if is_test:
            self.texts = df["masked_text"].tolist() if "masked_text" in df.columns else df.iloc[:, 1].tolist()
            self.ids = df["id"].tolist() if "id" in df.columns else df.iloc[:, 0].tolist()
            self.labels = None
        else:
            self.texts = df["masked_text"].tolist() if "masked_text" in df.columns else df.iloc[:, 1].tolist()
            if "predicted_text" in df.columns:
                self.labels = df["predicted_text"].tolist()
            elif "original_text" in df.columns:
                self.labels = df["original_text"].tolist()
            elif "target_words" in df.columns:
                self.labels = df["target_words"].tolist()
            else:
                self.labels = df.iloc[:, 2].tolist()
            self.ids = None

        self.tokenizer = tokenizer
        self.max_length = max_length
        self.model_type = model_type
        self.is_test = is_test

        if model_type == "bow":
            self._setup_bow_vectorizer()
        print(f"Dataset carregado: {len(self.texts)} amostras")

    def _setup_bow_vectorizer(self):
        self.vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
        all_texts = self.texts + self.labels if self.labels and not self.is_test else self.texts
        self.vectorizer.fit(all_texts)
        self.bow_vocab_size = len(self.vectorizer.get_feature_names_out())

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        if self.is_test:
            if self.model_type == "bow":
                input_vector = self.vectorizer.transform([text]).toarray()[0]
                return torch.tensor(input_vector, dtype=torch.float32), self.ids[idx]
            else:
                encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
                return encoding['input_ids'].squeeze(0), self.ids[idx]
        else:
            label = self.labels[idx]
            if self.model_type == "bow":
                input_vector = self.vectorizer.transform([text]).toarray()[0]
                label_vector = self.vectorizer.transform([label]).toarray()[0]
                return torch.tensor(input_vector, dtype=torch.float32), torch.tensor(label_vector, dtype=torch.float32)
            else:
                text_encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
                input_ids = text_encoding['input_ids'].squeeze(0)

                mask_token_id = getattr(self.tokenizer, 'mask_token_id', -1)
                if mask_token_id == -1 or mask_token_id is None:
                    try:
                        mask_token_id = self.tokenizer.convert_tokens_to_ids(self.tokenizer.mask_token)
                    except:
                        mask_token_id = -1

                if mask_token_id != -1:
                    mask_indices = (input_ids == mask_token_id).nonzero(as_tuple=True)[0]
                    mask_pos = mask_indices[0].item() if mask_indices.numel() > 0 else -1
                else:
                    mask_pos = -1

                label_tokens = self.tokenizer.encode(label, add_special_tokens=False)
                target_token_id = label_tokens[0] if len(label_tokens) > 0 else getattr(self.tokenizer, 'unk_token_id', 0)

                return input_ids, torch.tensor(target_token_id, dtype=torch.long), torch.tensor(mask_pos, dtype=torch.long)

def train_imdb_model(model, dataloader, criterion, optimizer, epochs=5, device=DEVICE):
    model.to(device)
    model.train()
    print(f"Iniciando treinamento por {epochs} épocas no device: {device}")
    for epoch in range(epochs):
        total_loss = 0
        batch_count = 0
        correct_predictions = 0
        total_samples = 0

        for batch_idx, batch_data in enumerate(dataloader):
            if len(batch_data) == 3:
                inputs, labels, _ = batch_data
            else:
                inputs, labels = batch_data

            inputs = inputs.to(device)
            labels = labels.to(device)

            model_name = model.__class__.__name__
            if 'Bow' in model_name or 'MLP' in model_name:
                inputs = inputs.to(torch.float32)
                labels = labels.to(torch.float32)
            else:
                inputs = inputs.to(torch.long)
                labels = labels.to(torch.long)

            optimizer.zero_grad()
            try:
                if hasattr(model, 'forward') and 'attention_mask' in model.forward.__code__.co_varnames:
                    attention_mask = (inputs != 0).to(torch.long)
                    outputs = model(inputs, attention_mask=attention_mask)
                else:
                    outputs = model(inputs)

                if 'Bow' in model_name or 'MLP' in model_name:
                    if len(outputs.shape) > 1 and outputs.shape[1] > 1:
                        outputs = outputs.mean(dim=1)
                    if len(labels.shape) > 1 and labels.shape[1] > 1:
                        labels = labels.mean(dim=1)
                    loss = criterion(outputs, labels)
                    with torch.no_grad():
                        if len(outputs.shape) == len(labels.shape):
                            diff = torch.abs(outputs - labels)
                            correct_predictions += (diff < 0.5).float().sum().item()
                            total_samples += labels.numel()
                else:
                    if len(outputs.shape) == 3:
                        outputs_flat = outputs.view(-1, outputs.shape[-1])
                        labels_flat = labels.view(-1)
                        mask = labels_flat != 0
                        loss = criterion(outputs_flat[mask], labels_flat[mask]) if mask.sum() > 0 else criterion(outputs_flat, labels_flat)
                        with torch.no_grad():
                            predicted = torch.argmax(outputs_flat, dim=-1)
                            if mask.sum() > 0:
                                correct_predictions += (predicted[mask] == labels_flat[mask]).sum().item()
                                total_samples += mask.sum().item()
                            else:
                                correct_predictions += (predicted == labels_flat).sum().item()
                                total_samples += labels_flat.numel()
                    else:
                        loss = criterion(outputs, labels)
                        with torch.no_grad():
                            predicted = torch.argmax(outputs, dim=-1) if len(outputs.shape) > 1 else torch.round(outputs)
                            correct_predictions += (predicted == labels).sum().item()
                            total_samples += labels.numel()

                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                batch_count += 1
            except Exception as e:
                continue

        avg_loss = total_loss / max(batch_count, 1)
        accuracy = (correct_predictions / max(total_samples, 1)) * 100
        print(f"Época {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, Acurácia: {accuracy:.2f}%")

def evaluate_imdb_model(model, dataloader, model_type="neural", device=DEVICE):
    model.to(device)
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    num_batches = 0

    print("🔍 Avaliando modelo...")
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if model_type == "bow" and len(batch) == 2:
                inputs, labels = batch
                mask_pos = torch.full((inputs.size(0),), -1, dtype=torch.long)
            else:
                inputs, labels, mask_pos = batch

            inputs = inputs.to(device)
            labels = labels.to(device)
            mask_pos = mask_pos.to(device)

            try:
                if model_type == "bow" or isinstance(model, BowModel):
                    inputs, labels = inputs.float(), labels.float()
                    outputs = model(inputs)
                    criterion = nn.MSELoss()
                    loss = criterion(outputs, labels)
                    similarity = torch.cosine_similarity(outputs, labels, dim=1)
                    correct_predictions += (similarity > 0.6).float().sum().item()
                    total_predictions += labels.size(0)
                elif isinstance(model, MLPModel) or isinstance(model, RNNModel):
                    inputs, labels = inputs.long(), labels.long()
                    outputs = model(inputs)
                    if len(outputs.shape) > 1 and outputs.shape[1] > 1:
                        outputs_flat = outputs.view(-1, outputs.shape[-1])
                        labels_flat = labels.view(-1)
                        criterion = nn.CrossEntropyLoss(ignore_index=0)
                        loss = criterion(outputs_flat, labels_flat)
                        predictions = torch.argmax(outputs, dim=-1)
                        correct_predictions += ((predictions == labels) & (labels != 0)).sum().item()
                        total_predictions += (labels != 0).sum().item()
                    else:
                        if len(labels.shape) > 1: labels = labels[:, 0]
                        criterion = nn.CrossEntropyLoss()
                        loss = criterion(outputs, labels)
                        correct_predictions += (torch.argmax(outputs, dim=-1) == labels).sum().item()
                        total_predictions += labels.size(0)
                elif isinstance(model, TransformerModel):
                    inputs, labels, mask_pos = inputs.long(), labels.long(), mask_pos.long()
                    attention_mask = (inputs != 0)
                    outputs = model(inputs, attention_mask=attention_mask)
                    predicted_logits_at_mask = outputs[torch.arange(outputs.size(0)), mask_pos]
                    criterion = nn.CrossEntropyLoss(ignore_index=-100)
                    loss = criterion(predicted_logits_at_mask, labels)
                    predictions = torch.argmax(predicted_logits_at_mask, dim=-1)
                    valid_mask_positions = (mask_pos != -1)
                    correct_predictions += (predictions[valid_mask_positions] == labels[valid_mask_positions]).sum().item()
                    total_predictions += valid_mask_positions.sum().item()

                total_loss += loss.item()
                num_batches += 1
            except:
                continue

    avg_loss = total_loss / max(num_batches, 1)
    accuracy = correct_predictions / max(total_predictions, 1e-9)
    print(f"📊 Resultados da Avaliação - Loss: {avg_loss:.4f}, Acurácia: {accuracy*100:.2f}%")
    return accuracy, avg_loss

def predict_masked_words(model, tokenizer, masked_text, model_type="neural", device=DEVICE):
    model.eval()
    model.to(device)
    mask_count = masked_text.count("<MASK>")
    if mask_count == 0: return ""
    try:
        with torch.no_grad():
            if model_type == "bow" or isinstance(model, BowModel):
                return ' '.join(['palavra'] * mask_count)
            else:
                encoding = tokenizer(masked_text, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
                input_ids = encoding['input_ids'].to(device)
                outputs = model(input_ids, attention_mask=encoding['attention_mask'].to(device)) if isinstance(model, TransformerModel) else model(input_ids)
                top_indices = torch.topk(outputs[0] if len(outputs.shape) > 1 and outputs.shape[1] > 1 else outputs.squeeze(), k=mask_count)[1]
                predicted_words = [tokenizer.decode([idx.item()], skip_special_tokens=True).strip() for idx in top_indices]
                return ' '.join([w if w else 'palavra' for w in predicted_words][:mask_count])
    except:
        return ' '.join(['palavra'] * max(mask_count, 1))

def generate_submission(model, tokenizer, test_df, sample_submission, model_type="neural", device=DEVICE):
    print("🔮 Fazendo predições nos dados de teste...")
    predictions = []
    for idx, row in test_df.iterrows():
        predicted_text = predict_masked_words(model, tokenizer, row['masked_text'], model_type, device)
        predictions.append({'id': row['id'], 'predicted_text': predicted_text})
    submission_df = pd.DataFrame(predictions).sort_values('id').reset_index(drop=True)
    submission_df.loc[(submission_df['predicted_text'] == '') | submission_df['predicted_text'].isna(), 'predicted_text'] = 'palavra'
    return submission_df

##Seção 5 (Markdown) — Modelos Disponíveis

In [ ]:
# ============================================================
# MODELOS DISPONÍVEIS — não altere esta célula
# ============================================================

class MLPModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim=64, output_dim=None):
        super(MLPModel, self).__init__()
        output_dim = vocab_size if output_dim is None else output_dim
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.fc1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.embedding(x).mean(dim=1) if len(x.shape) > 1 else self.embedding(x)
        return self.fc2(self.relu(self.fc1(x)))

class RNNModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128, num_layers=1, output_dim=None):
        super(RNNModel, self).__init__()
        output_dim = vocab_size if output_dim is None else output_dim
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.rnn = nn.RNN(hidden_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        rnn_out, _ = self.rnn(self.embedding(x))
        return self.fc(self.dropout(rnn_out.mean(dim=1)))

class RNNModel_FrozenGradients(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128, num_layers=1, output_dim=None):
        super(RNNModel_FrozenGradients, self).__init__()
        output_dim = vocab_size if output_dim is None else output_dim
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.rnn = nn.RNN(hidden_dim, hidden_dim, num_layers=8, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.9)
        for param in self.rnn.parameters():
            if param.dim() > 1: nn.init.constant_(param, 0.001)

    def forward(self, x):
        rnn_out, _ = self.rnn(self.embedding(x))
        return torch.tanh(self.fc(self.dropout(rnn_out[:, 0, :]))) * 0.01

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, num_heads=8, num_layers=6, ff_dim=2048, max_seq_len=512, output_dim=None, tie_weights=True):
        super(TransformerModel, self).__init__()
        output_dim = vocab_size if output_dim is None else output_dim
        self.vocab_size, self.embed_dim = vocab_size, embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = nn.Parameter(torch.randn(max_seq_len, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=0.1, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(0.1)
        self.output_projection = nn.Sequential(nn.Linear(embed_dim, ff_dim), nn.GELU(), nn.Dropout(0.1), nn.Linear(ff_dim, embed_dim), nn.GELU(), nn.Dropout(0.1), nn.Linear(embed_dim, output_dim))
        if tie_weights and isinstance(self.output_projection[-1], nn.Linear):
            if self.output_projection[-1].weight.shape == self.embedding.weight.shape:
                self.output_projection[-1].weight = self.embedding.weight

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.size()
        x = self.embedding(input_ids) * math.sqrt(self.embed_dim)
        x = self.dropout(x + self.pos_encoding[:T, :].unsqueeze(0).to(device=input_ids.device, dtype=x.dtype))
        h = self.transformer(x, src_key_padding_mask=(attention_mask == 0).to(device=input_ids.device, dtype=torch.bool) if attention_mask is not None else (input_ids == 0))
        return self.output_projection(self.layer_norm(h))

class BowModel(nn.Module):
    def __init__(self, vocab_size=5000, hidden_dim=32):
        super(BowModel, self).__init__()
        self.actual_vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.layers_initialized = True
        self.fc1 = nn.Linear(vocab_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x.float())))

# Função de inicialização que estava faltando nesta célula:
def get_model(model_name, tokenizer, device="cuda" if torch.cuda.is_available() else "cpu"):
    vocab_size = getattr(tokenizer, 'vocab_size', 10000)

    if model_name == "MLP":
        model = MLPModel(vocab_size=vocab_size, hidden_dim=128, output_dim=vocab_size)
        model_type = "neural"
    elif model_name == "RNN":
        vocab_size = tokenizer.vocab_size
        model = RNNModel(vocab_size=vocab_size, hidden_dim=128)
        model_type = "neural"
    elif model_name == "RNNModel_FrozenGradients":
        model = RNNModel_FrozenGradients(vocab_size=vocab_size, hidden_dim=64, num_layers=1, output_dim=vocab_size)
        model_type = "neural"
    elif model_name == "Transformer":
        vocab_size = tokenizer.vocab_size
        model = TransformerModel(vocab_size=vocab_size, embed_dim=512, num_heads=8, ff_dim=2048)
        model_type = "neural"
    elif model_name == "Bag of Words":
        vocab_size = bow_input_size
        model = BowModel(vocab_size=vocab_size, hidden_dim=32)
        model_type = "bow"
    else:
        print(f"Modelo '{model_name}' não reconhecido. Usando Transformer...")
        model = TransformerModel(vocab_size=vocab_size, embed_dim=256, num_heads=8, num_layers=4, ff_dim=512, max_seq_len=128, output_dim=vocab_size)
        model_type = "neural"

    model.to(device)
    print(f"Modelo criado: {model_name} | Device: {device} | Tipo: {model_type}")
    return model, model_type

##Seção 6 (Código) — Pipeline do Aluno (TODO)

In [ ]:
# ============================================================
# SEU CÓDIGO AQUI — Monte o pipeline de treinamento
# ============================================================

# PASSO 1: Escolha o Tokenizador modificando a string de entrada
tokenizador = "Byte Pair Encoding"  # Algoritmos: "WordPiece", "Byte Pair Encoding", "SentencePiece", "Turbo"
tokenizer = get_tokenizer(tokenizador)

# PASSO 2: Selecione o modelo configurando o nome correspondente
# Opções: "MLP", "RNN", "RNNModel_FrozenGradients", "Transformer", "Bag of Words"
modelo_escolhido = "Transformer"
model, model_type = get_model(modelo_escolhido, tokenizer, DEVICE)

# PASSO 3: Instancie os Datasets estruturados de forma consistente
dataset_train = IMDBTextDataset(train_subset, tokenizer, max_length=128, model_type=model_type, is_test=False)
dataset_val = IMDBTextDataset(val_subset, tokenizer, max_length=128, model_type=model_type, is_test=False)

# PASSO 4: Defina os hiperparâmetros de carregamento e otimização
batch = 4
epochs = 1
learning_rate = 0.00001

dataloader_train = DataLoader(dataset_train, batch_size=batch, shuffle=True)
dataloader_val = DataLoader(dataset_val, batch_size=batch, shuffle=False)

criterion = nn.MSELoss() if model_type == "bow" else nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# PASSO 5: Execute o loop de treinamento construído
train_imdb_model(model, dataloader_train, criterion, optimizer, epochs=epochs, device=DEVICE)

##Seção 7 (Código) — Avaliação

In [ ]:
# ============================================================
# AVALIAÇÃO — não altere
# ============================================================

print("\n🔍 --- AVALIAÇÃO DO MODELO ---")
final_accuracy, final_loss = evaluate_imdb_model(model, dataloader_val, model_type, DEVICE)

##Seção 8 (Código) — Geração do submission.csv

In [ ]:
# ============================================================
# EXPORTAÇÃO — não altere
# ============================================================

submission = generate_submission(model, tokenizer, test_df, sample_submission, model_type, DEVICE)
submission.to_csv(OUTPUT_FILE, index=False)

print(f"\n📁 Arquivo salvo: {OUTPUT_FILE}")
print(f"Linhas: {len(submission)} | Colunas: {list(submission.columns)}")

# Download automático no ambiente Colab (evitando NameError por escopo de célula)
from google.colab import files
files.download(OUTPUT_FILE)
print("Submissão baixada!")

##Seção 9 (Código) — Exploração Livre
# ============================================================
# EXPLORAÇÃO LIVRE — Células extras para o aluno (Opcional)
# ============================================================

# Use este espaço se quiser testar engenharia de atributos adicional,
# tunagem de hiperparâmetros ou novos algoritmos sem quebrar as células fixas acima.
